# 第72章 电商销售数据分析

以一份可复现的订单明细为例，完成从数据质量检查、指标计算到销售结论表达的完整分析闭环。

## 项目背景

某电商团队希望了解上半年销售额由哪些区域、渠道和品类贡献，并找出值得进一步复盘的订单结构。数据是教学用的模拟订单，不代表真实经营结果。

## 学习目标

- 建立订单明细数据字典
- 处理重复、缺失和异常金额
- 按区域、渠道和品类拆解销售表现
- 用静态图表支持业务结论


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| order_id | 订单编号 | 一行代表一笔订单 |
| order_date | 下单日期 | 订单发生日期 |
| region | 区域 | 华东、华南、华北、西南 |
| channel | 渠道 | 自然流量、广告、会员 |
| category | 品类 | 办公、数码、家居 |
| units | 购买件数 | 订单中的商品数量 |
| unit_price | 商品单价 | 单位：元 |
| discount | 折扣系数 | 1表示无折扣 |
| customer_id | 客户编号 | 用于估算客户数 |

## 数据质量检查清单

- 订单编号是否重复
- 关键字段是否缺失
- 购买件数和单价是否为正数
- 日期是否能正常解析
- 折扣系数是否位于0到1之间


## 项目任务

1. 生成并检查订单明细
2. 清理质量问题并计算订单金额
3. 比较区域、渠道和品类
4. 绘制销售额与订单量图表
5. 写出至少两条有数据依据的结论


## 1. 准备订单明细

先固定随机种子和字段含义，后续每次运行都能得到相同的教学数据。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(72)
n = 18
orders = pd.DataFrame({
    "order_id": [f"O{index:03d}" for index in range(1, n + 1)],
    "order_date": pd.date_range("2026-01-01", periods=n, freq="9D"),
    "region": rng.choice(["华东", "华南", "华北", "西南"], n),
    "channel": rng.choice(["自然流量", "广告", "会员"], n),
    "category": rng.choice(["办公", "数码", "家居"], n),
    "units": rng.integers(1, 6, n),
    "unit_price": rng.choice([59, 89, 129, 299, 499, 799], n),
    "discount": rng.choice([0.85, 0.9, 0.95, 1.0], n),
    "customer_id": [f"U{index:03d}" for index in rng.integers(1, 13, n)],
})

print("行数:", len(orders))
print(orders.head())


## 2. 检查并清洗质量问题

教学数据故意加入一个缺失价格和一条重复订单，先记录问题，再生成用于分析的干净表。


In [ ]:
raw_orders = orders.copy()
raw_orders.loc[3, "unit_price"] = np.nan
raw_orders = pd.concat([raw_orders, raw_orders.iloc[[0]]], ignore_index=True)

print("重复订单:", raw_orders["order_id"].duplicated().sum())
print("缺失值:\n", raw_orders.isna().sum())
print("非法件数:", (raw_orders["units"] <= 0).sum())

clean_orders = raw_orders.drop_duplicates(subset="order_id").copy()
clean_orders["unit_price"] = clean_orders["unit_price"].fillna(clean_orders["unit_price"].median())
clean_orders["order_date"] = pd.to_datetime(clean_orders["order_date"], errors="coerce")
clean_orders["amount"] = clean_orders["units"] * clean_orders["unit_price"] * clean_orders["discount"]
print("清洗后行数:", len(clean_orders))
print("销售额合计:", round(clean_orders["amount"].sum(), 2))


## 3. 拆解区域、渠道和品类

同一指标要保持统一口径，这里销售额使用折扣后的订单金额，客单价使用销售额除以订单数。


In [ ]:
region_summary = clean_orders.groupby("region").agg(
    sales=("amount", "sum"),
    orders=("order_id", "nunique"),
    customers=("customer_id", "nunique"),
).sort_values("sales", ascending=False)
region_summary["average_order_value"] = region_summary["sales"] / region_summary["orders"]

channel_summary = clean_orders.groupby("channel", as_index=False)["amount"].sum().sort_values("amount", ascending=False)
category_summary = clean_orders.groupby("category", as_index=False)["amount"].sum().sort_values("amount", ascending=False)
print("区域摘要:\n", region_summary.round(2))
print("\n渠道销售额:\n", channel_summary.round(2))
print("\n品类销售额:\n", category_summary.round(2))


## 4. 绘制销售分析图

左图比较区域销售额，右图观察按日期累计的销售趋势；图表标题和单位要能脱离代码独立阅读。


In [ ]:
daily = clean_orders.groupby("order_date", as_index=False)["amount"].sum().sort_values("order_date")
daily["cumulative_sales"] = daily["amount"].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
region_summary["sales"].sort_values().plot.barh(ax=axes[0], color="#1a73e8", title="各区域销售额")
axes[0].set_xlabel("销售额（元）")
axes[0].set_ylabel("区域")
axes[1].plot(daily["order_date"], daily["cumulative_sales"], marker="o", color="#188038")
axes[1].set(title="累计销售趋势", xlabel="日期", ylabel="累计销售额（元）")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
plt.show()

print("最高销售区域:", region_summary.index[0])
print("最高销售渠道:", channel_summary.iloc[0]["channel"])


## 5. 形成结论草稿

把图表观察转成可复核的结论，结论必须同时给出对象、指标和方向。


In [ ]:
top_region = region_summary.iloc[0]
top_category = category_summary.iloc[0]
repeat_customer_rate = clean_orders["customer_id"].duplicated().mean()
print(f"区域结论：{region_summary.index[0]}销售额最高，为 {top_region['sales']:.2f} 元。")
print(f"品类结论：{top_category['category']}贡献销售额 {top_category['amount']:.2f} 元。")
print(f"客户结论：重复购买订单占比约 {repeat_customer_rate:.1%}，可进一步分析复购结构。")


## 结论与表达

- 结论必须绑定清洗后的数据和明确指标口径
- 区域排名适合配合订单数和客单价一起解释
- 发现差异后应回到明细检查样本量和异常记录


## 项目验收清单

- Notebook从头运行不报错
- 包含数据字典和质量检查结果
- 至少完成一张比较图和一张趋势图
- 结论中出现具体数值
- 清洗前后行数和规则可追溯

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

以一份可复现的订单明细为例，完成从数据质量检查、指标计算到销售结论表达的完整分析闭环。


### 你已经完成

- 建立订单明细数据字典
- 处理重复、缺失和异常金额
- 按区域、渠道和品类拆解销售表现
- 用静态图表支持业务结论


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 生成并检查订单明细 |
| 步骤 2 | 清理质量问题并计算订单金额 |
| 步骤 3 | 比较区域、渠道和品类 |
| 步骤 4 | 绘制销售额与订单量图表 |
| 步骤 5 | 写出至少两条有数据依据的结论 |


### 质量与结论提醒

- 订单编号是否重复
- 关键字段是否缺失
- 购买件数和单价是否为正数
- 结论必须绑定清洗后的数据和明确指标口径
- 区域排名适合配合订单数和客单价一起解释
- 发现差异后应回到明细检查样本量和异常记录


### 项目交付检查

- [ ] Notebook从头运行不报错
- [ ] 包含数据字典和质量检查结果
- [ ] 至少完成一张比较图和一张趋势图
- [ ] 结论中出现具体数值
- [ ] 清洗前后行数和规则可追溯
